In [27]:
import pandas as pd
import numpy as np
import folium
import requests

from bs4 import BeautifulSoup

# pip install tqdm : progressBar 구현
from tqdm.notebook import tqdm 
from selenium import webdriver

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service

# jupyter nbconvert --to script coffeeStore.ipynb

In [28]:
chrome_options = webdriver.ChromeOptions() # 크롬 브라우저 옵션
drive_path = 'chromedriver.exe' # 다운로드 받은 드라이버 파일
myservice = Service(drive_path) # 드라이버 제어를 위한 서비스 객체
driver = webdriver.Chrome(service=myservice, options=chrome_options) # 드라이버 객체
print(type(driver)) # 객체가 잘 생성되었는 지 확인

wait_time = 10 # 최대 대기 시간 10초
driver.implicitly_wait(wait_time)

<class 'selenium.webdriver.chrome.webdriver.WebDriver'>


In [29]:
driver.maximize_window() # 윈도우 창 최대화

In [30]:
# 스타벅스) 매장 찾기-지역 검색 링크 주소
starbucks_url = 'https://www.starbucks.co.kr/store/store_map.do?disp=locale'
driver.get(starbucks_url) # 해당 페이지로 이동하기

In [31]:
# 스타벅스 '서울' 링크 클릭
starbucks_seoul_selector = '#container > div > form > fieldset > div > section > article.find_store_cont > article > article:nth-child(4) > div.loca_step1 > div.loca_step1_cont > ul > li:nth-child(1) > a'
WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CSS_SELECTOR, starbucks_seoul_selector))).click()

In [32]:
# 스타벅스 '서울'-'전체' 링크 클릭
starbucks_seoul_all = '#mCSB_2_container > ul > li:nth-child(1) > a'
WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CSS_SELECTOR, starbucks_seoul_all))).click()

In [33]:
# 스타벅스 html 코드를 파싱(parsing)해서 html 파일에 기록합니다. 
html = driver.page_source # 해당 페이지의 소스 코드 반환
filename = 'starbucks.html'
htmlFile = open(filename, mode='wt', encoding='UTF-8')
print(html, file=htmlFile)
htmlFile.close()
print(filename + ' 파일 생성됨')

starbucks.html 파일 생성됨


In [34]:
# 파싱된 결과를 BeautifulSoup 객체로 생성합니다. 
soup = BeautifulSoup(html, 'html.parser')
print(type(soup))

<class 'bs4.BeautifulSoup'>


In [35]:
container = soup.find('div', id='mCSB_3_container')
storeAll = container.find_all('li')
print(f'모든 매장 개수 : {len(storeAll)}')

모든 매장 개수 : 633


In [36]:
'''
모든 내용들은 white Character 문자열을 제거하도록 합니다.
상호명은 <strong> 태그 내의 문자열을 추출합니다.
주소는 <p> 태그 내의 문자열을 추출하되, '1522-3232'는 공백으로 치환합니다.
구(gu) 정보는 주소에서 추출하도록 합니다.
'''
starbucksData = []

for store in storeAll:
    # print(store)
    brand = '스타벅스'
    name = store.find('strong').text.strip()
    address = store.find('p').text.strip().replace('1522-3232', '')
    imsi = address.split(' ')
    # sido = imsi[0]
    gu = imsi[1]
    latitude = store['data-lat'] # 위도
    longitude = store['data-long'] # 경도
    
    starbucksData.append([brand, name, address, gu, latitude, longitude])
    # break
# end for

print(len(starbucksData))

633


In [37]:
'''
<li class="quickResultLstCon" data-code="3762" data-hlytag="null" data-index="0" 
        data-lat="37.501087" data-long="127.043069" data-name="역삼아레나빌딩" 
        data-storecd="1509" style="background:#fff"> 
    <strong data-my_siren_order_store_yn="N" 
        data-name="역삼아레나빌딩" data-store="1509" data-yn="N">역삼아레나빌딩  
    </strong> 
    <p class="result_details">서울특별시 강남구 언주로 425 (역삼동)
        <br/>1522-3232
    </p> 
    <i class="pin_general">리저브 매장 2번</i>
</li>
'''

'\n<li class="quickResultLstCon" data-code="3762" data-hlytag="null" data-index="0" \n        data-lat="37.501087" data-long="127.043069" data-name="역삼아레나빌딩" \n        data-storecd="1509" style="background:#fff"> \n    <strong data-my_siren_order_store_yn="N" \n        data-name="역삼아레나빌딩" data-store="1509" data-yn="N">역삼아레나빌딩  \n    </strong> \n    <p class="result_details">서울특별시 강남구 언주로 425 (역삼동)\n        <br/>1522-3232\n    </p> \n    <i class="pin_general">리저브 매장 2번</i>\n</li>\n'

In [39]:
sbDataFrame = pd.DataFrame(starbucksData)
sbDataFrame.columns = ['브랜드', '상호', '주소', '군구', '위도', '경도']
sbDataFrame.head()

,브랜드,상호,주소,군구,위도,경도
0,스타벅스,역삼아레나빌딩,서울특별시 강남구 언주로 425 (역삼동),강남구,37.501087,127.043069
1,스타벅스,논현역사거리,서울특별시 강남구 강남대로 538 (논현동),강남구,37.510178,127.022223
2,스타벅스,신사역성일빌딩,서울특별시 강남구 강남대로 584 (논현동),강남구,37.5139309,127.0206057
3,스타벅스,국기원사거리,서울특별시 강남구 테헤란로 125 (역삼동),강남구,37.499517,127.031495
4,스타벅스,대치재경빌딩,서울특별시 강남구 남부순환로 2947 (대치동),강남구,37.494668,127.062583


In [40]:
print('위도 누락 데이터 개수 : %d' % sbDataFrame['위도'].isnull().sum())
print('경도 누락 데이터 개수 : %d' % sbDataFrame['경도'].isnull().sum())

위도 누락 데이터 개수 : 0
경도 누락 데이터 개수 : 0


In [80]:
filename = 'starbucksFile.csv'
sbDataFrame.to_csv(filename, encoding='UTF-8', index=False)
print(f'{filename} 파일이 저장되었습니다.')

starbucksFile.csv 파일이 저장되었습니다.


In [41]:
guList = list(sbDataFrame['군구'].unique())
print('서울시 구 목록 개수 : %d' % len(guList))
print(guList)

서울시 구 목록 개수 : 25
['강남구', '강북구', '강서구', '관악구', '광진구', '금천구', '노원구', '도봉구', '동작구', '마포구', '서대문구', '서초구', '성북구', '송파구', '양천구', '영등포구', '은평구', '종로구', '중구', '강동구', '구로구', '동대문구', '성동구', '용산구', '중랑구']


In [42]:
# 이디야 매장 정보

In [43]:
ediya_url = 'https://www.ediya.com/contents/find_store.html'
driver.get(ediya_url) #  

In [44]:
# 이디야 '매장명'과 '주소' 검색에서 '주소' 클릭
ediya_address_selector = '#contentWrap > div.contents > div > div.store_search_pop > ul > li:nth-child(2) > a'
WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CSS_SELECTOR, ediya_address_selector))).click()

In [45]:
# 누락된 위도와 경도 정보는 kakao api를 이용하여 채워 넣도록 합니다.
# https://developers.kakao.com/
# kakao에 로그인하여 API 키 발급 받기

In [46]:
url_header = 'https://dapi.kakao.com/v2/local/search/address.json?query='
api_key = 'd2fb0b46a449c5e9e636a0078db80cc7'
header = {'Authorization': 'KakaoAK ' + api_key}

In [47]:
# 주소를 입력 받아서 위도와 경도를 반환해주는 함수 구현하기
def getGeoCoder(address):
    result = ""
    url = url_header + address
    response = requests.get(url, headers=header)
    # print(response) # 성공시 <Response [200]>으로 리턴 됨
    # print(response.json())
    if response.status_code == 200:
        try:
            result_address = response.json()["documents"][0]["address"]
            result = result_address["y"], result_address["x"]
        except Exception as err:
            print(err)
            return None
    else:
        result = "ERROR[" + str(response.status_code) + "]"

    return result
# end def

In [48]:
'''
위 경도 변환이 안되는 주소들
서울 노원구 한글비석로 409 (상계동) 1~2층
서울 동작구 사당로16가길 96 (사당동) 1,2층
서울 마포구 백범로 100 (대흥동)
서울 영등포구 영등포로35길 19 (영등포동6가)
'''
'''
{'errorType': 'NotAuthorizedError', 'message': 'App(커피매장) disabled OPEN_MAP_AND_LOCAL service.'}
'ERROR[403]' 라고 오류가 발생합니다.
'''
# 한 개의 매장에 대한 테스트를 진행합니다.
geoInfo = getGeoCoder('서울 중랑구 망우로 460 (망우동)') # 잘 동작하는 주소
# geoInfo = getGeoCoder('서울 노원구 한글비석로 409 (상계동) 1~2층') # NoneType이 리턴되는 주소
geoInfo

('37.6001065609187', '127.103136691889')

In [49]:
text = "abc\d"
text.index("c\d")

2

In [50]:
'''
매장 1개 정보 
<dt> 태그가 상호, <dd> 태그가 주소
<li class="item">
    <a href="#c" onclick="panLatTo('0','0','0');fnMove();">
    <div class="store_thum">
        <img src="../images/customer/store_thum.gif"/>
    </div>
    <dl>
        <dt>강남YMCA점</dt>
        <dd>서울 강남구 논현동</dd>
    </dl>
    </a>
</li>

위도/경도 추출 결과 : 'a' 태그의 'onclick'에 다음과 같은 값들이 들어 있습니다.
케이스 01
panLatTo('0','0','14');fnMove();

케이스 02
panLatTo('126.929036226245','37.489287127719','7');fnMove();

케이스 03
panAddTo('서울 관악구 보라매로 62 (봉천동, 보라매삼성아파트)','8');fnMove();
'''

'\n매장 1개 정보 \n<dt> 태그가 상호, <dd> 태그가 주소\n<li class="item">\n    <a href="#c" onclick="panLatTo(\'0\',\'0\',\'0\');fnMove();">\n    <div class="store_thum">\n        <img src="../images/customer/store_thum.gif"/>\n    </div>\n    <dl>\n        <dt>강남YMCA점</dt>\n        <dd>서울 강남구 논현동</dd>\n    </dl>\n    </a>\n</li>\n\n위도/경도 추출 결과 : \'a\' 태그의 \'onclick\'에 다음과 같은 값들이 들어 있습니다.\n케이스 01\npanLatTo(\'0\',\'0\',\'14\');fnMove();\n\n케이스 02\npanLatTo(\'126.929036226245\',\'37.489287127719\',\'7\');fnMove();\n\n케이스 03\npanAddTo(\'서울 관악구 보라매로 62 (봉천동, 보라매삼성아파트)\',\'8\');fnMove();\n'

In [65]:
# 위도와 경도 추출에 문제가 있는 주소 정보들
incorrect_address = []

In [66]:
eyidaData = [] # 이디야 매장 정보를 저장할 리스트

ediya_search_keyword_css = '#keyword' # 이디아 주소 검색 입력란
ediya_search_button_css = '#keyword_div > form > button' # 이디야 주소 검색(돋보기) 버튼

for gu in tqdm(guList):    
    # 이디야 주소 검색 입력란 초기화
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, ediya_search_keyword_css))).clear()
    
    # 이디야 주소 검색어 입력
    # f"서울 {gu}" : f-string 기법, 서울 중구, 서울 마포구 등등
    WebDriverWait(driver, 10).until(EC.presence_of_element_located
                                    ((By.CSS_SELECTOR, ediya_search_keyword_css))).send_keys(f'서울 {gu}')

    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, ediya_search_button_css))).click()

    # 이디야 매장 정보 수집
    html = driver.page_source

    if gu == '강남구': # 강남구만 저장해 보기
        # 개별 구에 대한 결과를 파일로 저장해 봅니다.
        filename = open(f"서울 {gu}.html", 'w', encoding='UTF-8')
        print(html, file=filename)
        filename.close()
        # print(f"서울 {gu}.html 파일 생성")
    # end if 

    soup = BeautifulSoup(html, 'html.parser')
    ul_tag = soup.find('ul', id='placesList')
    
    oneGuEdiyaList = ul_tag.find_all('li')
    for store in oneGuEdiyaList: 
        # print(store)
        brand = '이디야'
        name = store.find('dt').text.strip()
        address = store.find('dd').text.strip()
        imsi = address.split(' ')
        sido = imsi[0] 
        gungu = imsi[1]

        # 위도/경도 정보가 들어 있는 문자열 
        geoInfoString = store.find('a')['onclick'] 

        # # 중간에 값이 변경이 되므로 원본 데이터를 백업 해둠
        # geoInfoImsi = geoInfoString 

        # 넘파이의 nan으로 무의미한 데이터 만들기
        latitude = np.nan
        longitude = np.nan
        # latLong = ['0', '0']  # Initialize latLong to a default value
        
        if geoInfoString.startswith("panLatTo"): 
            # 올바른 위도/경도가 아님
            try:
                if geoInfoString.index(r"panLatTo('0','0'") == 0: 
                    # 올바른 위도/경도 형식이 아니므로, address를 사용하여 kakao API에게 물어 보기 
                    geoInfo = getGeoCoder(address) 
                    if geoInfo != None:
                        latitude = geoInfo[0] # 위도
                        longitude = geoInfo[1] # 경도
                    else:
                        incorrect_address.append(address)
                        latitude = None  # 차후 보정 작업 예정
                        longitude = None            
            except ValueError as err:
                # 올바른 위도/경도입니다.
                latLong = geoInfoString.replace("panLatTo('", "").replace(");fnMove();", "")
                latLong = latLong.split("','")

                # 주의) '경도'가 먼저 나오고 '위도'가 뒤쪽에 나옴
                latitude = latLong[1] 
                longitude = latLong[0]                
            # end try

        else: # 'panAddTo'으로 시작하는 경우
            geoInfo = getGeoCoder(address) 
            if geoInfo != None:
                latitude = geoInfo[0] # 위도
                longitude = geoInfo[1] # 경도
            else:
                incorrect_address.append(address)
                latitude = None
                longitude = None
        # end if

        # # 올바른 위도 경도 형식이 아니면
        # if latLong[1] == '0' or latLong[0] == '0':            
        #     # 카카오 지도 api 이용하여 위도와 경도를 취득합니다.
        #     geoInfo = getGeoCoder(address) 
        #     if geoInfo != None:
        #         latitude = geoInfo[0] # 위도
        #         longitude = geoInfo[1] # 경도
        #     else:
        #         incorrect_address.append(address)
        #         latitude = None
        #         longitude = None
        # # end if
        
        eyidaData.append([brand, name, address, gungu, latitude, longitude])
        # break
    # end inner for
    # break
# end outer for 

print('이디야 매장 개수 : %d' % len(eyidaData))

  0%|          | 0/25 [00:00<?, ?it/s]

list index out of range
list index out of range
list index out of range
이디야 매장 개수 : 549


In [67]:
ediyaDataFrame = pd.DataFrame(eyidaData)
ediyaDataFrame.columns = ['브랜드', '상호', '주소', '군구', '위도', '경도']
ediyaDataFrame.head()

,브랜드,상호,주소,군구,위도,경도
0,이디야,금란망우점,서울 중랑구 망우로 460 (망우동),중랑구,37.6001065609187,127.103136691889
1,이디야,동원사거리점,"서울 중랑구 겸재로 240 (면목동, 행복오피스텔)",중랑구,37.5896269575279,127.094182772191
2,이디야,망우중앙점,"서울 중랑구 용마산로115길 109 (망우동, 한일써너스빌리젠시2단지)",중랑구,37.5974674047065,127.09415879594557
3,이디야,망우코레일점,"서울 중랑구 망우로55길 11-10 (상봉동, 망우역)",중랑구,37.5992876153903,127.092756577852
4,이디야,먹골역점,서울 중랑구 동일로157길 13 (묵동),중랑구,37.6097411262067,127.076897926489


In [77]:
print('위도 누락 데이터 개수 : %d' % ediyaDataFrame['위도'].isnull().sum())
print('경도 누락 데이터 개수 : %d' % ediyaDataFrame['경도'].isnull().sum())

위도 누락 데이터 개수 : 0
경도 누락 데이터 개수 : 0


In [73]:
incorrect_address # 변환에 문제가 있었던 주소 목록

['서울 노원구 한글비석로 409 (상계동) 1~2층',
 '서울 동작구 사당로16가길 96 (사당동) 1,2층',
 '서울 영등포구 영등포로35길 19 (영등포동6가)']

In [74]:
# 주소 각각에 대하여 콤마를 스페이스로 치환하고, 공백 2칸도 1칸으로 치환합니다.
# split() 메소드를 사용한 후, 0번째 3번째 까지 챙겨 옵니다.
correction_address = [addr.replace(',', ' ').replace('  ', ' ').split(' ')[0:4] for addr in incorrect_address]
# split() 함수로 나누어진 주소를 취합하여 보정된 주소로 변환합니다.
correction_address = [' '.join(addr) for addr in correction_address]
# 주소 정보와 위도 및 경도를 묶어서 하나의 tuple로 만든 목록들을 list에 저장합니다.
correction_data = [(incorrect_address[idx], correction_geoInfo[idx][0], correction_geoInfo[idx][1]) 
                  for idx in range(len(incorrect_address))]
correction_data

[('서울 노원구 한글비석로 409 (상계동) 1~2층', '37.6516375000425', '127.012463182729'),
 ('서울 동작구 사당로16가길 96 (사당동) 1,2층', '37.5502242501914', '126.863458548999'),
 ('서울 영등포구 영등포로35길 19 (영등포동6가)', '37.5071964496082', '127.091386305975')]

In [75]:
# 반복문으로 각 행에 대한 보정 작업
# onerow는 (주소, 위도, 경도) 정보를 담고 있는 tuple입니다.
for onerow in correction_data:
    ediyaDataFrame.loc[ediyaDataFrame['주소'] == onerow[0], ['위도', '경도']] = [onerow[1], onerow[2]]

In [78]:
print('위도 누락 데이터 개수 : %d' % ediyaDataFrame['위도'].isnull().sum())
print('경도 누락 데이터 개수 : %d' % ediyaDataFrame['경도'].isnull().sum())

위도 누락 데이터 개수 : 0
경도 누락 데이터 개수 : 0


In [79]:
# 이디야 매장 csv 파일로 저장
filename = 'ediyaFile.csv'
# ediyaDataFrame.to_csv(filename, encoding='CP949', index=False)
ediyaDataFrame.to_csv(filename, encoding='UTF-8', index=False)
print(filename + ' 파일 저장됨')

ediyaFile.csv 파일 저장됨


In [ ]:
# 할리스 매장

In [ ]:
'''
<img alt="테라스" src="https://www.hollys.co.kr/websrc/images/store/img_store_s02.gif" style="margin-right:1px"/>
<img alt="흡연시설" src="https://www.hollys.co.kr/websrc/images/store/img_store_s04.gif" style="margin-right:1px"/>
<img alt="주차" src="https://www.hollys.co.kr/websrc/images/store/img_store_s08.png" style="margin-right:1px"/>
<img src="https://www.hollys.co.kr/websrc/images/store/img_store_s06.gif" style="margin-right:1px" alt="24시간">
<img src="https://www.hollys.co.kr/websrc/images/store/img_store_s01.gif" style="margin-right:1px" alt="DT 매장">
'''

In [ ]:
hollys_base_url = 'https://www.hollys.co.kr/store/korea/korStore2.do'

In [ ]:
# 위도와 경도 추출에 문제가 있는 주소 정보들
incorrect_address = []

# 서비스 목록 이름을 저장해 놓은 전역 변수 : 타입은 list
global_service_list = []

In [ ]:
# 페이지  번호를 추적하여 매장의 정보를 읽어 들이는 함수
def get_store_data(page_no):
    # 요청할 때 전송할 파라미터 정보
    # 페이지 번호와 '서울' 지역만 조회 예정
    parameters = {
        'pageNo': page_no,
        'sido': '서울',
        'gugun': '', 
        'store': ''        
    }    
    
    # url에 데이터 요청 보내기
    response = requests.get(hollys_base_url, params=parameters)

    if response.status_code == 200 :        
        # print(response.text)
        pass
    else :
        print(f'페이지 {page_no} 로딩에 실패하였습니다.')
        return [] 
    # end if

    soup = BeautifulSoup(response.text, 'html.parser') # HTML 문서 파싱
    # table 태그의 'class' 속성이 'tb_store'인 요소를 찾습니다.
    store_table = soup.find('table', {'class': 'tb_store'})
    
    all_store = store_table.find_all('tr')[1:] # 상단 table header는 제외하기  

    if len(all_store) == 0 : # 매장 정보가 없으면 ...
        return None

    stores = [] # 현재 페이지에서 반환될 모든 매장 정보        

    for one_store in all_store :
        td_tags = one_store.find_all('td')

        if len(td_tags) <= 1 : # 더 이상 매장 정보가 없으면 무시
            continue
        # end if
        
        brand = '할리스'
        # region = td_tags[0].text.strip()
        name = td_tags[1].text.strip()
        address = td_tags[3].text.strip().replace(' .', '') # 주소 끝트러미에 "공백." 치환
        imsi = address.split(' ')
        # sido = imsi[0]
        gungu = imsi[1]

        # 매장 서비스 목록 : 가능한 서비스들을 <img> 태그로 만들었습니다.
        # 기본 값으로 모든 매장에서는 서비스 되는 게 없습니다.
        service_dict = {'테라스':'no', '흡연시설':'no', '주차':'no', '24시간':'no', 'DT 매장':'no'}
        
        store_service = td_tags[4] # .text.strip() 
        # print('-' * 30)
        # print(store_service)

        service_img = store_service.find_all('img')
        # print('-' * 30)
        # print(f'img 태그 개수 : {len(service_img)}')

        for image in service_img:
            alt_tag = image.get('alt', '')
            service_dict[alt_tag] = 'yes'

        # print(name)
        # print(service_dict)

        # 사전을 정렬(숫자, 알파벳, 한글 순)하여 각, key들의 값 정보를 추출합니다.      
        service_list = sorted(service_dict.keys())
        service_data = [service_dict[key] for key in service_list]
        # print(service_data) # stores list에 추가되어야 합니다.

        global global_service_list # 나는 전역 변수입니다.
        global_service_list = service_list
        
        phone_number = td_tags[5].text.strip() # 전화 번호
        if phone_number == '.': # 전화 번호에 점만 들어 있는 경우
            phone_number = None

        # 위도와 경도에 대한 정보가 없습니다.
        # 그래서, 주소 정보와 kakao API를 이용하여 구해야 합니다.
        geoInfo  = getGeoCoder(address)
        # print('geoInfo')
        # print(geoInfo)

        if geoInfo != None :
            latitude = geoInfo[0]
            longitude = geoInfo[1]
            
        else : # 위도와 경도 변환이 잘 안되는 주소들
            # print(address)
            incorrect_address.append(address)
            latitude = None
            longitude = None
        # end if   
        
        # list에 + 기호를 사용하면, 요소들을 합쳐줍니다.
        stores.append([brand, name, address, gungu, latitude, longitude, phone_number] + service_data)
    # end for  

    return stores
# end def get_store_data(page_no)    

In [ ]:
hollysData = [] # 매장 정보를 저장할 리스트
page_no = 1 # 페이지 번호를 +1씩 증가하면서 크롤링

while True :
    print(f'현재 {page_no} 페이지를 크롤링 중입니다.')
    stores = get_store_data(page_no)

    if not stores :
    # if page_no == 5 :
        break # 더 이상 매장 정보가 없으므로 빠져 나가기

    hollysData.extend(stores)
    page_no += 1    
# end while True    

In [ ]:
# 크롤링한 데이터를 DataFrame으로 변환
hollysFrame = pd.DataFrame(hollysData)
print(f'총 {len(hollysFrame)} 개의 매장 정보가 수집되었습니다.')

In [ ]:
column_list = ['브랜드', '상호', '주소', '군구', '위도', '경도', '전화 번호'] + global_service_list
hollysFrame.columns = column_list
hollysFrame.head()

In [ ]:
# 크롤링 이후 생성된 데이터 프레임에 대한 보정 작업

In [ ]:
# 변환에 문제가 있었던 주소들
incorrect_address

In [ ]:
# 주소 각각에 대하여 콤마를 스페이스로 치환하고, 공백 2칸도 1칸으로 치환합니다.
# split() 메소드를 사용한 후, 0번째 3번째 까지 챙겨 옵니다.
correction_address = [addr.replace(',', ' ').replace('  ', ' ').split(' ')[0:4] for addr in incorrect_address]
correction_address

In [ ]:
# split() 함수로 나누어진 주소를 취합하여 보정된 주소로 변환합니다.
correction_address = [' '.join(addr) for addr in correction_address]
correction_address

In [ ]:
# 보정된 주소 이름으로 getGeoCoder 함수를 호출하여 위도와 경도를 추출해 냅니다.
correction_geoInfo = [getGeoCoder(addr) for addr in correction_address]
correction_geoInfo

In [ ]:
# 주소 정보와 위도 및 경도를 묶어서 하나의 tuple로 만든 목록들을 list에 저장합니다.
correction_data = [(incorrect_address[idx], correction_geoInfo[idx][0], correction_geoInfo[idx][1]) 
                  for idx in range(len(incorrect_address))]
correction_data

In [ ]:
# 반복문으로 각 행에 대한 보정 작업
# onerow는 (주소, 위도, 경도) 정보를 담고 있는 tuple입니다.
for onerow in correction_data:
    hollysFrame.loc[hollysFrame['주소'] == onerow[0], ['위도', '경도']] = [onerow[1], onerow[2]]

In [81]:
hollysFrame.loc[hollysFrame['전화 번호'] == '없음', ['전화 번호']] = [None]

In [82]:
filename = 'hollysFile.csv'
hollysFrame.to_csv(filename, encoding='UTF-8', index=False)
print(f'{filename} 파일이 저장되었습니다.')

hollysFile.csv 파일이 저장되었습니다.
